In [7]:
import sys
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import geopandas as gpd
from shapely.geometry import mapping
from scipy.stats import spearmanr, pearsonr
import pandas as pd
import gc

import warnings
warnings.filterwarnings('ignore')

In [8]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
dataf = "/Users/ellendyer/Documents/GitHub/F4R_data/"

In [9]:
tropess_in = xr.open_dataset('/Users/ellendyer/Documents/GitHub/F4R_data/tropess_regrid/tropess_gridded_caf_multl_2018_2024.nc')
tropess_in = tropess_in.sel(lat=slice(-15,12),lon=slice(8,31),drop=True).load()
tropess_in = tropess_in.sortby('lat', ascending=True)
tropess_in = tropess_in.sortby('lon', ascending=True)
tropess_in = tropess_in.interp(lat=np.arange(tropess_in["lat"].min().values,tropess_in["lat"].max().values,0.25), lon=np.arange(tropess_in["lon"].min().values,tropess_in["lon"].max().values,0.25), method="linear")

tropess_all_list = []
for Y in range(2018,2025):
    tropess = tropess_in.sel(time=slice(str(Y)+'-01-01',str(Y)+'-12-31'))
    tropess_year_list = []
    for m in range(1,13):
        try:
            mp = tropess.sel(time=(tropess.time.dt.month==m), drop=True)
            print(mp)
            bins = [mp.time[0].values,mp.time[10-1].values,mp.time[20-1].values,mp.time[-1].values]
            mp_out = mp.groupby_bins('time', bins,labels=[mp.time[10-1].values,mp.time[20-1].values,mp.time[-1].values]).mean()
            mp_out = mp_out.rename({'time_bins':'time'})
            #print(mp_out)
            tropess_year_list.append(mp_out)
        except:
            print('no month - ',m,' for year - ',Y)
    tropess_year = xr.concat(tropess_year_list,dim='time')
    tropess_all_list.append(tropess_year)
    print('done - ',Y)
tropess_all = xr.concat(tropess_all_list,dim='time')
tropess_all = tropess_all.sel(time=slice('2018-07-01','2024-12-31'))

tropess_all.to_netcdf(dataf+'tropess_10day_reg_regrid.nc',engine='h5netcdf')
        

<xarray.Dataset> Size: 33MB
Dimensions:  (time: 30, level: 7, lat: 107, lon: 91)
Coordinates:
  * time     (time) datetime64[ns] 240B 2018-01-01 2018-01-02 ... 2018-01-31
  * level    (level) int32 28B 825 750 680 620 510 420 350
  * lat      (lat) float64 856B -15.0 -14.75 -14.5 -14.25 ... 11.0 11.25 11.5
  * lon      (lon) float64 728B 8.0 8.25 8.5 8.75 9.0 ... 29.75 30.0 30.25 30.5
Data variables:
    deltaD   (time, level, lat, lon) float64 16MB nan nan nan ... nan nan nan
    H2O      (time, level, lat, lon) float64 16MB nan nan nan ... nan nan nan
Attributes:
    description:  deltaD
    units:        permil
<xarray.Dataset> Size: 27MB
Dimensions:  (time: 25, level: 7, lat: 107, lon: 91)
Coordinates:
  * time     (time) datetime64[ns] 200B 2018-02-01 2018-02-02 ... 2018-02-28
  * level    (level) int32 28B 825 750 680 620 510 420 350
  * lat      (lat) float64 856B -15.0 -14.75 -14.5 -14.25 ... 11.0 11.25 11.5
  * lon      (lon) float64 728B 8.0 8.25 8.5 8.75 9.0 ... 29.75 30.0 3